In [1]:
#!/usr/bin/env python3

"""
An example MDMC script for optimizing Lennard Jones parameters for liquid Ar.
For info on syntax see the MDMC docs, including the jupyter notebook tutorials.
A copy of the data fitting against is assumed to be located in
../doc/tutorials/data/Well_s_q_omega_Ar_data.xml
"""

import numpy as np
import os
# Change the number of threads depending on the number of physical cores on
# your computer as it was tested for LAMMPS
os.environ["OMP_NUM_THREADS"] = "4"
os.environ["OMPI_MCA_btl"]="^vader"
from scipy.interpolate import interp2d

from MDMC.control import Control
from MDMC.MD import Atom, LennardJones, Simulation, Universe
from MDMC.MD.interactions import Dispersion

# Build universe with density 0.0176 atoms per AA^-3
density = 0.0176
# This means cubic universe of side:
# 23.0668 A will contain 216 Ar atoms
# 26.911 A will contain 343 Ar atoms
# 30.7553 A will contain 512 Ar atoms
# 38.4441 A will contain 1000 Ar atoms
universe = Universe(dimensions=38.4441)
Ar = Atom('Ar', charge=0.)
# Calculating number of Ar atoms needed to obtain density
n_ar_atoms = int(density * np.product(universe.dimensions))
universe.fill(Ar, num_struc_units=(n_ar_atoms))

# Above an universe of non-interacting argon atoms was created. Below
# specify how these atoms will interact
Ar_dispersion = Dispersion(universe,
                           (Ar.atom_type, Ar.atom_type),
                           cutoff=8.0,
                           vdw_tail_correction=True,
                           function=LennardJones(1.0243, 3.36))

# MD Engine setup. time_step of 10 fs is somewhat high, but for argon OK-ish.
# If time_step is descreased by a factor consider increasing traj_step by the
# same factor.

N_STEPS=4


Supported DL_POLY version 5.0
Universe created with:
  Dimensions       [38.44, 38.44, 38.44]
  Force field                       None
  Number of atoms                      0



In [2]:
exp_datasets = [{'file_name':'data/Well_s_q_omega_Ar_data.xml',
                 'type':'SQw',
                 'reader':'xml_SQw',
                 'weight':1.,
                 'resolution':2500.0,
                 'auto_scale':True}]

In [3]:
simulation_dlp = Simulation(universe,
                        engine="dlpoly",
                        time_step=10.18893,
                        temperature=120.,
                        traj_step=15,
                        numprocs=4)

# Energy Minimization and equilibration
simulation_dlp.minimize(n_steps=10,output_log='minim.log',work_dir='minim')
simulation_dlp.run(n_steps=1000, equilibration=True,output_log='equilibration.log',work_dir='equil')
#simulation.run(n_steps=10000, equilibration=False)
#print(simulation.trajectory)
## dataset


fit_parameters = universe.parameters

# Specify how the refinement is going to be controlled
control_dlp = Control(simulation=simulation_dlp,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  auto_scale=True,
                  MD_steps=5700)

# Run the refinement, i.e. refine the FF parameters against the data.
# n_steps = 3 is too small, but a good choice to first test this script
control_dlp.refine(n_steps=N_STEPS)


# same factor.


Simulation created with dlpoly engine and settings:
  temperature  120.0
  numprocs       4.0

Folder minim exists, over-writing.
update coordinates from  /workspaces/MDMCv02_pilot/doc/tutorials/minim/REVCON
Folder equil exists, over-writing.


KeyboardInterrupt: 

In [ ]:
simulation_lmp = Simulation(universe,
                        engine="lammps",
                        time_step=10.18893,
                        temperature=120.,
                        traj_step=15,
                        numprocs=4)

# Energy Minimization and equilibration
simulation_lmp.minimize(n_steps=10,output_log='minim.log',work_dir='minim')
simulation_lmp.run(n_steps=1000, equilibration=True,output_log='equilibration.log',work_dir='equil')
#simulation.run(n_steps=10000, equilibration=False)
#print(simulation.trajectory)


# Specify how the refinement is going to be controlled
control_lmp = Control(simulation=simulation_lmp,
                  exp_datasets=exp_datasets,
                  fit_parameters=fit_parameters,
                  auto_scale=True,
                  MD_steps=5700)

# Run the refinement, i.e. refine the FF parameters against the data.
# n_steps = 3 is too small, but a good choice to first test this script
control_lmp.refine(n_steps=N_STEPS)



In [ ]:
obs_pair_dlp = control_dlp.observable_pairs[0]
obs_pair_lmp = control_lmp.observable_pairs[0]

In [ ]:
obs_pair_lmp.rescale_factor

In [ ]:
result_dlp = obs_pair_dlp.MD_obs.SQw / obs_pair_dlp.rescale_factor
result_lmp = obs_pair_lmp.MD_obs.SQw / obs_pair_lmp.rescale_factor

In [ ]:

%matplotlib widget
from MDMC.trajectory_analysis.observables.obs_factory import ObservableFactory
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.widgets import Slider
obs=ObservableFactory.create_observable('SQw')
obs.read_from_file(reader='xml_SQw', file_name='/workspaces/MDMCv0.2_pilot/doc/tutorials/data/Well_s_q_omega_Ar_data.xml')
SQw=obs.SQw[0]
SQw_err=obs.SQw_err[0]
Q=obs.Q
E=obs.E
fig, ax = plt.subplots()
line, = ax.plot(E, SQw[1], linewidth=1, color='black')
SQw_lmp = result_lmp
line_lmp, = ax.plot(E, SQw_lmp[0,1,:], linewidth=1, color='blue')
SQw_dlp = result_dlp
line_dlp, = ax.plot(E, SQw_dlp[0,1,:], linewidth=1, color='red')
# ax.set_xlim([-1, 1])
ax.set_xlabel('E (meV)')
ax.set_ylabel('S(Q,E) (arb)')
ax.set_title('Argon data')
fig.subplots_adjust(left=0.25, bottom=0.3)
Q_slider_ax  = fig.add_axes([0.25, 0.15, 0.65, 0.03], facecolor='lightgoldenrodyellow')
Q_slider = Slider(Q_slider_ax, 'Q index', 0, len(Q)-1, valinit=1, valstep=1)
Q_label=plt.text(1,1.7,f'Q={Q[1]} $\AA^{-1}$')
def Q_on_changed(val):
    line.set_ydata(SQw[val])
    line_lmp.set_ydata(SQw_lmp[0,val])
    line_dlp.set_ydata(SQw_dlp[0,val])
    Q_label.set_text(f'Q={Q[val]} $\AA^{-1}$')
    fig.canvas.draw_idle()
    ax.set_ylim(0,max(np.max(SQw_lmp[0,val]),1e-5))
Q_slider.on_changed(Q_on_changed)
plt.show()